# Row-aware WL/FRL particle-row comparison

Run the cell below after setting the STAR file paths and options in the input block.

In [1]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

from scipy.signal import find_peaks
from scipy.spatial import cKDTree
from scipy.spatial.distance import pdist
from sklearn.neighbors import NearestNeighbors


# =============================================================================
# USER SETTINGS
# =============================================================================

# Input STAR files. These may be raw particle lists or already row-assigned lists.
STAR_WL = "particles.star"
STAR_FRL = "particles.star"

LABEL_WL = "wl-PBS"
LABEL_FRL = "frl-PBS"

# RELION/Warp columns.
TOMO_COL = "_rlnTomoName"
COORD_COLS = [
    "_rlnCenteredCoordinateXAngst",
    "_rlnCenteredCoordinateYAngst",
    "_rlnCenteredCoordinateZAngst",
]
EULER_COLS = ["_rlnAngleRot", "_rlnAngleTilt", "_rlnAnglePsi"]
ROW_COL = "RowID"

# Row assignment mode.
# - If True, use an existing RowID/_rlnRowID column if present.
# - If False, always recompute rows from coordinates and Euler angles.
USE_EXISTING_ROWIDS_IF_PRESENT = True

# Same graph-clustering parameters as the PBS row-assignment notebook.
SPATIAL_TH = 160.0       # Å; maximum inter-particle distance for a row edge
ANGLE_TH_ROT = 20.0      # degrees; maximum absolute ΔRot
ANGLE_TH_TILT = 10.0     # degrees; maximum absolute ΔTilt
ANGLE_TH_PSI = 20.0      # degrees; maximum absolute ΔPsi
K_NEIGHBORS = 5          # nearest neighbors evaluated per particle
MIN_ROW_SIZE = 2         # rows smaller than this are excluded from row-aware metrics

# Within-row pair-distance / g(r) parameters.
R_MAX_ANG = 3000.0
DR_ANG = 25.0
PEAK_PROMINENCE_FRACTION = 0.08
PEAK_DISTANCE_BINS = 4
N_PEAKS_TO_ANNOTATE = 3

# Histogram bins.
ROW_LENGTH_BINS = np.arange(1, 51, 1)
ANGLE_BINS_DEG = np.arange(0, 91, 2)
CENTROID_DISTANCE_BINS_ANG = np.arange(0, 3001, 50)

# Plot style.
WL_COLOR = "#3986a1ff"
FRL_COLOR = "#a41858ff"
COLOR_BY_CONDITION = {LABEL_WL: WL_COLOR, LABEL_FRL: FRL_COLOR}
FIGSIZE_SINGLE = (6.2, 4.4)
FIGSIZE_WIDE = (7.0, 4.4)
DPI = 300

# Optional saving. Default is display-only.
SAVE_FIGURES = False
SAVE_TABLES = False
OUTPUT_DIR = Path("row_comparison_publication_outputs")
FIGURE_FORMATS = ("png", "pdf")


# =============================================================================
# STAR parsing
# =============================================================================


def _make_unique(names: Sequence[str]) -> List[str]:
    """Make duplicate STAR column names unique by appending __dupN."""
    counts: Dict[str, int] = {}
    out: List[str] = []
    for name in names:
        n = counts.get(name, 0)
        out.append(name if n == 0 else f"{name}__dup{n}")
        counts[name] = n + 1
    return out


def _collapse_duplicate_columns_keep_last(df: pd.DataFrame) -> pd.DataFrame:
    """
    Collapse duplicate STAR columns generated by _make_unique.

    For duplicated RELION labels, the final occurrence is retained. This handles
    files where corrected columns such as _rlnRowID or _rlnRandomSubset were
    appended later.
    """
    base_to_cols: Dict[str, List[str]] = {}
    for col in df.columns:
        base = col.split("__dup")[0]
        base_to_cols.setdefault(base, []).append(col)

    out = pd.DataFrame(index=df.index)
    for base, cols in base_to_cols.items():
        out[base] = df[cols[-1]]
    return out


def read_star_loop(path: str | Path, preferred_block: str = "data_particles") -> pd.DataFrame:
    """Read a RELION-style STAR loop into a DataFrame."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"STAR file not found: {path}")

    lines = path.read_text().splitlines()

    block_idx: Optional[int] = None
    target = preferred_block.lower()
    for i, line in enumerate(lines):
        if line.strip().lower() == target:
            block_idx = i
            break

    if block_idx is None:
        for i, line in enumerate(lines):
            if line.strip().lower().startswith("data_"):
                block_idx = i
                break

    if block_idx is None:
        raise ValueError(f"No data_ block found in {path}")

    loop_idx: Optional[int] = None
    for i in range(block_idx + 1, len(lines)):
        if lines[i].strip().lower() == "loop_":
            loop_idx = i
            break

    if loop_idx is None:
        raise ValueError(f"No loop_ found after data block in {path}")

    raw_cols: List[str] = []
    data_start: Optional[int] = None

    for i in range(loop_idx + 1, len(lines)):
        txt = lines[i].strip()
        if not txt:
            continue
        if txt.startswith("_"):
            raw_cols.append(txt.split()[0])
        else:
            data_start = i
            break

    if not raw_cols or data_start is None:
        raise ValueError(f"Could not parse STAR columns/data in {path}")

    cols = _make_unique(raw_cols)
    rows: List[List[str]] = []

    for line in lines[data_start:]:
        txt = line.strip()
        if not txt or txt.startswith("#"):
            continue
        if txt.startswith("data_") or txt == "loop_" or txt.startswith("_"):
            break
        parts = txt.split()
        if len(parts) == len(cols):
            rows.append(parts)

    if not rows:
        raise ValueError(f"No particle rows parsed from {path}")

    df = pd.DataFrame(rows, columns=cols)
    return _collapse_duplicate_columns_keep_last(df)


def standardize_particles(df: pd.DataFrame, condition: str) -> pd.DataFrame:
    """Validate required columns and convert coordinates/angles to numeric."""
    missing = [c for c in [TOMO_COL, *COORD_COLS, *EULER_COLS] if c not in df.columns]
    if missing:
        raise ValueError(
            f"{condition}: missing required columns: {missing}\n"
            f"Available columns: {list(df.columns)}"
        )

    out = df.copy()
    for col in [*COORD_COLS, *EULER_COLS]:
        out[col] = pd.to_numeric(out[col], errors="coerce")

    out = out.dropna(subset=[TOMO_COL, *COORD_COLS, *EULER_COLS]).copy()
    out["Condition"] = condition
    return out.reset_index(drop=True)


def find_existing_rowid_column(df: pd.DataFrame) -> Optional[str]:
    """Return the most likely existing row-ID column, if present."""
    candidates = [ROW_COL, "_rlnRowID", "_rlnRowId", "_rlnRowid", "rowID", "row_id"]
    for col in candidates:
        if col in df.columns:
            return col
    return None


# =============================================================================
# Row assignment
# =============================================================================


def angular_difference_abs(e1: np.ndarray, e2: np.ndarray) -> Tuple[float, float, float]:
    """
    Absolute Euler-angle differences used by the original PBS row-assignment script.

    This intentionally uses component-wise absolute differences rather than full
    rotation-matrix distances, because the goal is to reproduce the original row
    clustering approach exactly.
    """
    d = np.abs(e1 - e2)
    return float(d[0]), float(d[1]), float(d[2])


def passes_angle_threshold(e1: np.ndarray, e2: np.ndarray) -> bool:
    """Return True if component-wise Euler differences pass all thresholds."""
    d_rot, d_tilt, d_psi = angular_difference_abs(e1, e2)
    return (
        d_rot <= ANGLE_TH_ROT
        and d_tilt <= ANGLE_TH_TILT
        and d_psi <= ANGLE_TH_PSI
    )


def assign_rows_by_graph(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Assign row IDs independently within each tomogram using graph clustering.

    Edges are accepted when particles are close in 3D and pass the component-wise
    Euler-angle gate. Connected components with more than one particle are rows.
    Singletons remain RowID = -1.
    """
    out = df.copy().reset_index(drop=True)
    out[ROW_COL] = -1

    edge_records: List[dict] = []
    global_row_id = 0

    for tomo, sub in out.groupby(TOMO_COL, sort=False):
        coords = sub[COORD_COLS].to_numpy(dtype=float)
        eulers = sub[EULER_COLS].to_numpy(dtype=float)
        idxs = sub.index.to_numpy()

        if len(sub) < 2:
            continue

        k_eff = min(K_NEIGHBORS, len(sub))
        neigh = NearestNeighbors(n_neighbors=k_eff).fit(coords).kneighbors(coords, return_distance=False)

        graph = nx.Graph()
        graph.add_nodes_from(idxs)

        for i_local, neighbors in enumerate(neigh):
            for j_local in neighbors[1:]:
                dist = float(np.linalg.norm(coords[i_local] - coords[j_local]))
                if dist > SPATIAL_TH:
                    continue

                if not passes_angle_threshold(eulers[i_local], eulers[j_local]):
                    continue

                ii = int(idxs[i_local])
                jj = int(idxs[j_local])
                graph.add_edge(ii, jj)

                d_rot, d_tilt, d_psi = angular_difference_abs(eulers[i_local], eulers[j_local])
                edge_records.append(
                    {
                        "Condition": sub["Condition"].iloc[0],
                        TOMO_COL: tomo,
                        "i": ii,
                        "j": jj,
                        "distance_A": dist,
                        "dRot_deg": d_rot,
                        "dTilt_deg": d_tilt,
                        "dPsi_deg": d_psi,
                        "angular_score_deg": float(np.sqrt(d_rot**2 + d_tilt**2 + d_psi**2)),
                    }
                )

        for component in nx.connected_components(graph):
            if len(component) > 1:
                for ii in component:
                    out.at[ii, ROW_COL] = global_row_id
                global_row_id += 1

    edge_df = pd.DataFrame(edge_records)
    return out, edge_df


def use_existing_rowids(df: pd.DataFrame, condition: str) -> pd.DataFrame:
    """Copy an existing row-ID column to ROW_COL."""
    row_col = find_existing_rowid_column(df)
    if row_col is None:
        raise ValueError(f"{condition}: no existing RowID/_rlnRowID column found.")

    out = df.copy()
    out[ROW_COL] = pd.to_numeric(out[row_col], errors="coerce").fillna(-1).astype(int)
    return out


def prepare_condition(df: pd.DataFrame, condition: str) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Load or assign row IDs for one condition."""
    if USE_EXISTING_ROWIDS_IF_PRESENT and find_existing_rowid_column(df) is not None:
        out = use_existing_rowids(df, condition)
        # Existing rows have no graph-edge table; within-row angular distribution is computed later.
        edge_df = pd.DataFrame()
    else:
        out, edge_df = assign_rows_by_graph(df)

    row_sizes = out.groupby([TOMO_COL, ROW_COL]).size().rename("RowSize")
    out = out.join(row_sizes, on=[TOMO_COL, ROW_COL])
    return out.reset_index(drop=True), edge_df


# =============================================================================
# Row-aware measurements
# =============================================================================


def rows_only(df: pd.DataFrame) -> pd.DataFrame:
    """Return only particles assigned to rows of at least MIN_ROW_SIZE."""
    return df[(df[ROW_COL] != -1) & (df["RowSize"] >= MIN_ROW_SIZE)].copy()


def compute_within_row_pair_distances(df_all: pd.DataFrame) -> pd.DataFrame:
    """All pairwise 3D distances among particles assigned to the same row."""
    records: List[dict] = []
    df_rows = rows_only(df_all)

    for (condition, tomo, row_id), sub in df_rows.groupby(["Condition", TOMO_COL, ROW_COL], sort=False):
        coords = sub[COORD_COLS].to_numpy(dtype=float)
        if len(coords) < 2:
            continue
        d = pdist(coords)
        records.extend(
            {
                "Condition": condition,
                TOMO_COL: tomo,
                ROW_COL: row_id,
                "pair_distance_A": float(x),
            }
            for x in d
        )

    return pd.DataFrame(records)


def compute_within_row_radial_summary(pair_df: pd.DataFrame) -> pd.DataFrame:
    """Convert within-row pair distances into probability and approximate g(r)."""
    edges = np.arange(0.0, R_MAX_ANG + DR_ANG, DR_ANG)
    centers = 0.5 * (edges[:-1] + edges[1:])
    shell_vol = (4.0 / 3.0) * np.pi * (edges[1:] ** 3 - edges[:-1] ** 3)

    rows: List[dict] = []

    for condition, sub in pair_df.groupby("Condition", sort=False):
        distances = pd.to_numeric(sub["pair_distance_A"], errors="coerce").dropna().to_numpy()
        hist, _ = np.histogram(distances, bins=edges)
        probability = hist / hist.sum() if hist.sum() else np.full_like(hist, np.nan, dtype=float)

        # Approximate comparative RDF normalization.
        # This is not an absolute RDF for a homogeneous 3D point process, because the
        # measurements are restricted to quasi-1D row structures.
        expected = shell_vol / np.nansum(shell_vol)
        g_r = np.divide(
            probability,
            expected,
            out=np.full_like(probability, np.nan, dtype=float),
            where=expected > 0,
        )

        for r, h, p, g in zip(centers, hist, probability, g_r):
            rows.append(
                {
                    "Condition": condition,
                    "r_A": float(r),
                    "pair_count": int(h),
                    "pair_probability": float(p) if np.isfinite(p) else np.nan,
                    "g_r": float(g) if np.isfinite(g) else np.nan,
                }
            )

    return pd.DataFrame(rows)


def compute_row_lengths(df_all: pd.DataFrame) -> pd.DataFrame:
    """Number of particles per assigned row."""
    df_rows = rows_only(df_all)
    return (
        df_rows.groupby(["Condition", TOMO_COL, ROW_COL], sort=False)
        .size()
        .rename("row_length")
        .reset_index()
    )


def compute_within_row_angles(df_all: pd.DataFrame, edge_df: pd.DataFrame | None = None) -> pd.DataFrame:
    """
    Within-row angular-neighbor distribution.

    For publication comparisons, this function reconstructs a comparable
    within-row neighbor graph for every condition, even when existing RowID
    columns were provided and the original graph edges are unavailable.

    For each assigned row, particles are connected only to same-row spatial
    nearest neighbors. Angular differences are then calculated only for these
    within-row neighbor pairs. This avoids mixing neighboring-edge angles for
    one condition with all pairwise within-row angles for another condition.
    """
    records: List[dict] = []
    df_rows = rows_only(df_all)

    for (condition, tomo, row_id), sub in df_rows.groupby(["Condition", TOMO_COL, ROW_COL], sort=False):
        n = len(sub)
        if n < 2:
            continue

        coords = sub[COORD_COLS].to_numpy(dtype=float)
        eulers = sub[EULER_COLS].to_numpy(dtype=float)

        # Same-row spatial neighbor graph. n_neighbors includes self.
        k_eff = min(K_NEIGHBORS + 1, n)
        nbrs = NearestNeighbors(n_neighbors=k_eff).fit(coords)
        _, neigh = nbrs.kneighbors(coords)

        seen_edges = set()
        for i_local, nbr_idx in enumerate(neigh):
            for j_local in nbr_idx[1:]:
                a, b = sorted((int(i_local), int(j_local)))
                if (a, b) in seen_edges:
                    continue
                seen_edges.add((a, b))

                distance = float(np.linalg.norm(coords[a] - coords[b]))
                d_rot, d_tilt, d_psi = angular_difference_abs(eulers[a], eulers[b])

                records.append(
                    {
                        "Condition": condition,
                        TOMO_COL: tomo,
                        ROW_COL: row_id,
                        "distance_A": distance,
                        "dRot_deg": d_rot,
                        "dTilt_deg": d_tilt,
                        "dPsi_deg": d_psi,
                        "angular_score_deg": float(np.sqrt(d_rot**2 + d_tilt**2 + d_psi**2)),
                    }
                )

    return pd.DataFrame(records)

def compute_centroid_spacing(df_all: pd.DataFrame) -> pd.DataFrame:
    """Nearest centroid-centroid spacing between rows within the same tomogram."""
    df_rows = rows_only(df_all)

    centroids = (
        df_rows.groupby(["Condition", TOMO_COL, ROW_COL], sort=False)[COORD_COLS]
        .mean()
        .reset_index()
    )

    records: List[dict] = []
    for (condition, tomo), sub in centroids.groupby(["Condition", TOMO_COL], sort=False):
        coords = sub[COORD_COLS].to_numpy(dtype=float)
        row_ids = sub[ROW_COL].to_numpy()
        if len(coords) < 2:
            continue

        tree = cKDTree(coords)
        dist, nbr = tree.query(coords, k=2)
        for i in range(len(coords)):
            records.append(
                {
                    "Condition": condition,
                    TOMO_COL: tomo,
                    ROW_COL: row_ids[i],
                    "nearest_row_id": row_ids[nbr[i, 1]],
                    "centroid_nn_distance_A": float(dist[i, 1]),
                }
            )

    return pd.DataFrame(records)


# =============================================================================
# Printing and plotting
# =============================================================================


def setup_matplotlib() -> None:
    """Set conservative publication-style matplotlib defaults."""
    plt.rcParams.update(
        {
            "figure.dpi": 120,
            "savefig.dpi": DPI,
            "font.size": 10,
            "axes.titlesize": 11,
            "axes.labelsize": 10,
            "xtick.labelsize": 9,
            "ytick.labelsize": 9,
            "legend.fontsize": 9,
            "axes.linewidth": 0.8,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
        }
    )


def color_for(condition: str) -> str:
    return COLOR_BY_CONDITION.get(condition, "black")


def polish_axis(ax: plt.Axes) -> None:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(direction="out", length=3, width=0.8)


def save_figure(fig: plt.Figure, stem: str) -> None:
    if not SAVE_FIGURES:
        return
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    for fmt in FIGURE_FORMATS:
        fig.savefig(OUTPUT_DIR / f"{stem}.{fmt}", bbox_inches="tight", dpi=DPI)


def save_table(df: pd.DataFrame, name: str) -> None:
    if not SAVE_TABLES:
        return
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    df.to_csv(OUTPUT_DIR / f"{name}.csv", index=False)


def print_section(title: str, explanation: str) -> None:
    line = "=" * len(title)
    print(f"\n{title}\n{line}\n{explanation}\n")


def describe_by_condition(df: pd.DataFrame, value_col: str, title: str) -> pd.DataFrame:
    if df.empty:
        print(f"{title}: no values.")
        return pd.DataFrame()

    summary = (
        df.groupby("Condition")[value_col]
        .agg(["count", "mean", "std", "median", "min", "max"])
        .reset_index()
    )
    print(title)
    print(summary.to_string(index=False))
    return summary


def annotate_peaks(
    ax: plt.Axes,
    x: np.ndarray,
    y: np.ndarray,
    condition: str,
    color: str,
) -> List[float]:
    """Detect and annotate strongest peaks on a line plot."""
    y_clean = np.nan_to_num(y, nan=0.0, posinf=0.0, neginf=0.0)
    if len(y_clean) < 3 or np.nanmax(y_clean) <= 0:
        return []

    prominence = PEAK_PROMINENCE_FRACTION * np.nanmax(y_clean)
    peaks, _ = find_peaks(y_clean, prominence=prominence, distance=PEAK_DISTANCE_BINS)
    if len(peaks) == 0:
        return []

    strongest = sorted(peaks, key=lambda i: y_clean[i], reverse=True)[:N_PEAKS_TO_ANNOTATE]
    strongest = sorted(strongest)

    peak_positions: List[float] = []
    for idx in strongest:
        peak_x = float(x[idx])
        peak_y = float(y_clean[idx])
        peak_positions.append(peak_x)
        ax.axvline(peak_x, color=color, linestyle="--", linewidth=0.8, alpha=0.35)
        ax.annotate(
            f"{condition}: {peak_x:.0f} Å",
            xy=(peak_x, peak_y),
            xytext=(4, 6),
            textcoords="offset points",
            ha="left",
            va="bottom",
            fontsize=8,
            color=color,
            rotation=90,
        )
    return peak_positions


def plot_line_metric(
    df: pd.DataFrame,
    y_col: str,
    ylabel: str,
    title: str,
    filename_stem: str,
) -> pd.DataFrame:
    fig, ax = plt.subplots(figsize=FIGSIZE_WIDE)
    peak_records: List[dict] = []

    for condition, sub in df.groupby("Condition", sort=False):
        sub = sub.sort_values("r_A")
        x = sub["r_A"].to_numpy(dtype=float)
        y = sub[y_col].to_numpy(dtype=float)
        color = color_for(condition)

        ax.plot(x, y, color=color, linewidth=1.8, label=condition)
        peaks = annotate_peaks(ax, x, y, condition, color)
        peak_records.extend({"Condition": condition, "metric": y_col, "peak_A": p} for p in peaks)

    ax.set_xlabel("Within-row pair distance (Å)")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(frameon=False)
    polish_axis(ax)
    fig.tight_layout()
    save_figure(fig, filename_stem)
    plt.show()

    peak_df = pd.DataFrame(peak_records)
    if peak_df.empty:
        print("No peaks detected with the current prominence settings.")
    else:
        print("Detected peak distances:")
        print(peak_df.to_string(index=False))
    return peak_df


def plot_histogram_metric(
    df: pd.DataFrame,
    value_col: str,
    bins: Iterable[float],
    xlabel: str,
    title: str,
    filename_stem: str,
    density: bool = True,
) -> None:
    fig, ax = plt.subplots(figsize=FIGSIZE_SINGLE)

    for condition, sub in df.groupby("Condition", sort=False):
        values = pd.to_numeric(sub[value_col], errors="coerce").dropna().to_numpy()
        if len(values) == 0:
            continue

        color = color_for(condition)
        ax.hist(
            values,
            bins=bins,
            density=density,
            histtype="stepfilled",
            alpha=0.28,
            color=color,
            edgecolor=color,
            linewidth=1.4,
            label=f"{condition} (n={len(values)})",
        )
        median = float(np.median(values))
        ax.axvline(median, color=color, linestyle="--", linewidth=1.1, alpha=0.9)

    ax.set_xlabel(xlabel)
    ax.set_ylabel("Density" if density else "Count")
    ax.set_title(title)
    ax.legend(frameon=False)
    polish_axis(ax)
    fig.tight_layout()
    save_figure(fig, filename_stem)
    plt.show()


# =============================================================================
# Main workflow
# =============================================================================


def run_analysis() -> None:
    setup_matplotlib()

    raw_wl = read_star_loop(STAR_WL)
    raw_frl = read_star_loop(STAR_FRL)

    wl = standardize_particles(raw_wl, LABEL_WL)
    frl = standardize_particles(raw_frl, LABEL_FRL)

    wl, wl_edges = prepare_condition(wl, LABEL_WL)
    frl, frl_edges = prepare_condition(frl, LABEL_FRL)

    df_all = pd.concat([wl, frl], ignore_index=True)
    edge_all = pd.concat([wl_edges, frl_edges], ignore_index=True) if not (wl_edges.empty and frl_edges.empty) else pd.DataFrame()

    row_lengths = compute_row_lengths(df_all)
    pair_distances = compute_within_row_pair_distances(df_all)
    radial_summary = compute_within_row_radial_summary(pair_distances)
    within_row_angles = compute_within_row_angles(df_all, edge_all)
    centroid_spacing = compute_centroid_spacing(df_all)

    # Optional table export.
    save_table(row_lengths, "row_length_distribution")
    save_table(pair_distances, "within_row_pair_distances")
    save_table(radial_summary, "within_row_radial_summary")
    save_table(within_row_angles, "within_row_angular_distribution")
    save_table(centroid_spacing, "centroid_centroid_interrow_spacing")

    print_section(
        "Dataset summary",
        "Particles are loaded independently for WL and FRL. Rows are assigned per tomogram using the same "
        "spatial-plus-angular graph-clustering logic as the original PBS row-assignment notebook. "
        "Singleton particles remain RowID = -1 and are excluded from all row-aware distributions.",
    )

    summary = []
    for condition, table in [(LABEL_WL, wl), (LABEL_FRL, frl)]:
        assigned = rows_only(table)
        summary.append(
            {
                "Condition": condition,
                "particles_total": len(table),
                "tomograms": table[TOMO_COL].nunique(),
                "particles_in_rows": len(assigned),
                "singletons": int((table[ROW_COL] == -1).sum()),
                "rows": assigned[[TOMO_COL, ROW_COL]].drop_duplicates().shape[0],
            }
        )
    summary_df = pd.DataFrame(summary)
    print(summary_df.to_string(index=False))
    save_table(summary_df, "dataset_summary")

    print_section(
        "1. Within-row pair-distance distribution",
        "This plot uses only particle pairs that belong to the same assigned row. It therefore reports "
        "row-internal repeat distances and avoids mixing within-row, between-row, and tomogram-scale distances.",
    )
    describe_by_condition(pair_distances, "pair_distance_A", "Within-row pair distances (Å)")
    plot_line_metric(
        radial_summary,
        y_col="pair_probability",
        ylabel="Pair-distance probability",
        title="Within-row pair-distance distribution",
        filename_stem="01_within_row_pair_distance_distribution",
    )

    print_section(
        "2. Approximate within-row g(r)",
        "The same within-row pair distances are shown after shell-volume normalization. Because PBS rows are "
        "quasi-1D/curvilinear structures embedded in 3D, g(r) is treated as a comparative normalization rather "
        "than an absolute radial distribution function for a homogeneous 3D point process.",
    )
    plot_line_metric(
        radial_summary,
        y_col="g_r",
        ylabel="Approximate within-row g(r)",
        title="Approximate within-row radial distribution function",
        filename_stem="02_approximate_within_row_gr",
    )

    print_section(
        "3. Row length distribution",
        "Row length is the number of particles in each connected component after graph clustering. Only rows "
        f"with at least {MIN_ROW_SIZE} particles are included.",
    )
    describe_by_condition(row_lengths, "row_length", "Row lengths (particles per row)")
    plot_histogram_metric(
        row_lengths,
        value_col="row_length",
        bins=ROW_LENGTH_BINS,
        xlabel="Particles per row",
        title="Row length distribution",
        filename_stem="03_row_length_distribution",
    )

    print_section(
        "4. Within-row angular distribution",
        "Angular differences are calculated from same-row spatial nearest-neighbor pairs. This keeps the "
        "comparison row-restricted and uses the same angular-neighbor definition for WL and FRL, even when "
        "rows were provided as existing RowID columns rather than rebuilt from graph edges.",
    )
    describe_by_condition(within_row_angles, "angular_score_deg", "Within-row angular score (degrees)")
    plot_histogram_metric(
        within_row_angles,
        value_col="angular_score_deg",
        bins=ANGLE_BINS_DEG,
        xlabel="Within-row angular score (degrees)",
        title="Within-row angular-neighbor distribution",
        filename_stem="04_within_row_angular_distribution",
    )
    
    for comp, col in [("Rot", "dRot_deg"), ("Tilt", "dTilt_deg"), ("Psi", "dPsi_deg")]:
        describe_by_condition(within_row_angles, col, f"Within-row Δ{comp} (degrees)")
        plot_histogram_metric(
            within_row_angles,
            value_col=col,
            bins=ANGLE_BINS_DEG,
            xlabel=f"Within-row Δ{comp} (degrees)",
            title=f"Within-row Δ{comp} distribution",
            filename_stem=f"04_within_row_d{comp.lower()}_distribution",
        )
        
    print_section(
        "5. Centroid-centroid inter-row spacing",
        "For each row, the centroid is calculated from its assigned particles. The plotted value is the nearest "
        "row-centroid distance within the same tomogram, preventing distances between unrelated tomograms or "
        "global coordinate frames from entering the distribution.",
    )
    describe_by_condition(centroid_spacing, "centroid_nn_distance_A", "Nearest row-centroid distances (Å)")
    plot_histogram_metric(
        centroid_spacing,
        value_col="centroid_nn_distance_A",
        bins=CENTROID_DISTANCE_BINS_ANG,
        xlabel="Nearest row-centroid distance (Å)",
        title="Centroid-centroid inter-row spacing",
        filename_stem="05_centroid_centroid_interrow_spacing",
    )

    if SAVE_FIGURES or SAVE_TABLES:
        print(f"\nSaved outputs to: {OUTPUT_DIR.resolve()}")


if __name__ == "__main__":
    run_analysis()


FileNotFoundError: STAR file not found: WL_K3_particles.star